# LexChain LLM document-analysis comparison (human-judged)

Three models from **different families**, one fixed prompt, on 10 OHR-Bench Law documents. Each model outputs a **summary + entities (parties/dates/monetary_amounts/obligations) + risk_flags** as JSON. The deliverable is a **blind human-scoring sheet** + an aggregation that produces `model | summary coverage | accuracy | fluency | entity F1 | risk F1`.

| # | Model | Family |
|---|---|---|
| 1 | `meta/llama-3.1-70b-instruct` | Meta (deployed) |
| 2 | `qwen/qwen2.5-72b-instruct` | Alibaba/Qwen |
| 3 | `mistralai/mixtral-8x22b-instruct-v0.1` | Mistral |

Provider: **NVIDIA NIM** (OpenAI-compatible, one provider for all three). Everything caches to Google Drive (`MyDrive/lexchain_bench/analysis`) and is resumable — a disconnect loses nothing.

**Set your key first:** Colab left sidebar → 🔑 Secrets → add `NVIDIA_API_KEY` (from build.nvidia.com), toggle notebook access on.

In [ ]:
# Cell 1 — mount Drive, clone repo, install deps, load key
from google.colab import drive, userdata
drive.mount('/content/drive')

import os
if not os.path.exists('/content/lexchain-chunk-embed-bench'):
    !git clone https://github.com/MarvinPescos/lexchain-chunk-embed-bench.git /content/lexchain-chunk-embed-bench
%cd /content/lexchain-chunk-embed-bench
!git pull
!pip install -q -r requirements-analysis.txt

os.environ['NVIDIA_API_KEY'] = userdata.get('NVIDIA_API_KEY')  # from Colab Secrets
print('key loaded:', bool(os.environ.get('NVIDIA_API_KEY')))

In [ ]:
# Cell 2 — ensure OHR-Bench Law ground truth is present (idempotent)
!python download_data.py

In [ ]:
# Cell 2b — ground-truth authoring materials (no API/model needed; do this early)
# Writes the BLANK hand-authored key template + readable dumps of the 10 docs.
# You can start filling ground_truth_key.csv while the model run happens.
!python -m analysis.prepare_ground_truth
print('\nDownload from MyDrive/lexchain_bench/analysis/ :')
print('  doc_texts/                     -> read these 10 .txt files')
print('  ground_truth_key_template.csv  -> hand-fill, save as ground_truth_key.csv')
print('  ground_truth_key_INSTRUCTIONS.txt')

In [ ]:
# Cell 3 — smoke test: 2 docs x 3 models (6 calls). Validates NIM + model IDs,
# reports per-call latency, and projects the full 30-call run time.
!python -m analysis.analyze --smoke 2

import json, glob
from pathlib import Path
cache = Path('/content/drive/MyDrive/lexchain_bench/analysis/analyses')
recs = [json.loads(p.read_text()) for p in cache.glob('*.json')]
lat = [r['latency_s'] for r in recs if r.get('latency_s')]
ok = sum(r.get('ok', False) for r in recs)
if lat:
    mean = sum(lat)/len(lat)
    print(f"\n{ok}/{len(recs)} parsed OK. mean latency {mean:.1f}s/call.")
    print(f"Projected full run: 30 calls x {mean:.1f}s ~= {30*mean/60:.1f} min "
          f"(free tier ~40 req/min, so rate limits are not a concern).")

**STOP — check the smoke output above** (all 6 parsed OK, latency reasonable) before the full run. If a model ID isn't served, Cell 3 prints the closest matches and stops; adjust `analysis/data.py` and re-run.

In [ ]:
# Cell 4 — full run: 10 docs x 3 models (resumable; smoke docs are reused)
!python -m analysis.analyze

In [ ]:
# Cell 5 — build the blind human-scoring scaffold
!python -m analysis.build_blind_eval
print('\nDownload these from MyDrive/lexchain_bench/analysis/ :')
print('  blind_eval.csv                 -> raters fill the 1-5 columns (identities hidden)')
print('  ground_truth_key_template.csv  -> you fill true entities/risks per doc,')
print('                                    then SAVE it as ground_truth_key.csv')
print('  unblinding_key.csv             -> DO NOT open until rating is done')

### Human step (off-Colab)
1. Rate all 30 rows in `blind_eval.csv` (`1`–`5`, 5 best) — you can hand ~40 rows to teammates to spot-check inter-rater agreement.
2. Fill `ground_truth_key_template.csv` with the true parties/dates/amounts/obligations/risk clauses per document and save it as **`ground_truth_key.csv`** (same folder).
3. Put both filled files back in `MyDrive/lexchain_bench/analysis/`, then run Cell 6.

In [ ]:
# Cell 6 — aggregate: un-blind, compute per-model ratings + entity/risk F1, write results
!python -m analysis.aggregate_analysis